### Read experimental result data

In [1]:
# read parquet file
import polars as pl
# df = pl.read_parquet("../../experiments/16_10_result.parquet")
df = pl.read_parquet("../../experiments/twitter_81306/result.parquet")
print(df)

# read metadata of the parquet file
import pyarrow.parquet as pq
# meta = pq.read_metadata("../test/result.parquet")
meta = pq.read_metadata("../../experiments/twitter_81306/result.parquet")

# You can check versions depending on libraries 
print(meta.metadata)

shape: (163_840, 5)
┌─────┬───────┬───────┬───────┬───────────────┐
│ id  ┆ ip    ┆ us_id ┆ msg   ┆ mean_num_xact │
│ --- ┆ ---   ┆ ---   ┆ ---   ┆ ---           │
│ u64 ┆ u64   ┆ u64   ┆ str   ┆ f64           │
╞═════╪═══════╪═══════╪═══════╪═══════════════╡
│ 0   ┆ 9199  ┆ 0     ┆ 11000 ┆ 47.71         │
│ 0   ┆ 9199  ┆ 0     ┆ 21000 ┆ 67.36         │
│ 0   ┆ 9199  ┆ 0     ┆ 31000 ┆ 159.64        │
│ 0   ┆ 9199  ┆ 0     ┆ 30000 ┆ 210.77        │
│ 0   ┆ 9199  ┆ 0     ┆ 20000 ┆ 197.07        │
│ …   ┆ …     ┆ …     ┆ …     ┆ …             │
│ 159 ┆ 68985 ┆ 15    ┆ 03233 ┆ 2.81          │
│ 159 ┆ 68985 ┆ 15    ┆ 13333 ┆ 2.49          │
│ 159 ┆ 68985 ┆ 15    ┆ 23333 ┆ 3.06          │
│ 159 ┆ 68985 ┆ 15    ┆ 33333 ┆ 3.04          │
│ 159 ┆ 68985 ┆ 15    ┆ 03333 ┆ 2.8           │
└─────┴───────┴───────┴───────┴───────────────┘
{b'version_runner': b'0.1.1', b'version_core': b'0.1.1', b'version': b'0.1.0', b'ARROW:schema': b'/////0wBAAAQAAAAAAAKAAwACgAJAAQACgAAABAAAAAAAQQACAAIAAAABAAIAAAABA

### Compute solutions and/or values of optimal, RAM, and URS policies

In [2]:
EV_BY_OPT = "ev_by_opt"
EV_BY_RAM = "ev_by_ram"
EV_BY_URS = "ev_by_urs"
OPT_MEAN = "mean_of_opt_values"
RAMEV_OPTEV_RATIO = "ram ev / opt ev"
URSEV_OPTEV_RATIO = "urs ev / opt ev"
RAMEV_OPTM_RATIO = "ram ev / opt mean"
URSEV_OPTM_RATIO = "urs ev / opt mean"

MEAN_NUM_XACT = "mean_num_xact"

lf_opt = (
    df.lazy()
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max().over(["us_id", "ip"]))
    .unique(["ip", "us_id", MEAN_NUM_XACT])
    .select([pl.col("us_id"), pl.col("ip"), pl.col("msg").name.prefix("argmax_"), pl.col(MEAN_NUM_XACT).name.prefix("max_")])
)
lf_opt_mod = (df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg"))
    .group_by("ip_right", "us_id_right")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .join(lf_opt, left_on=["ip_right", "us_id_right"], right_on=["ip", "us_id"])
    .select([MEAN_NUM_XACT, "max_" + MEAN_NUM_XACT])
    .mean()
    .select([pl.col(MEAN_NUM_XACT).alias(EV_BY_OPT), pl.col("max_" + MEAN_NUM_XACT).alias(OPT_MEAN)])
)
lf_rob = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max())
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_RAM))
)
lf_urs = (
    df.lazy()
    .select(MEAN_NUM_XACT)
    .mean()
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_URS))
)
eval_df = (
    pl.concat([lf_opt_mod, lf_rob, lf_urs], how="horizontal")
    .with_columns([
        (pl.col(EV_BY_RAM) / pl.col(EV_BY_OPT)).alias(RAMEV_OPTEV_RATIO),
        (pl.col(EV_BY_URS) / pl.col(EV_BY_OPT)).alias(URSEV_OPTEV_RATIO),
        (pl.col(EV_BY_RAM) / pl.col(OPT_MEAN)).alias(RAMEV_OPTM_RATIO),
        (pl.col(EV_BY_URS) / pl.col(OPT_MEAN)).alias(URSEV_OPTM_RATIO),
    ])
).collect().transpose(include_header=True).rename({"column": "key", "column_0": "value"})

In [3]:
eval_df

key,value
str,f64
"""ev_by_opt""",1484.187414
"""mean_of_opt_values""",2076.590688
"""ev_by_ram""",1971.692625
"""ev_by_urs""",666.908027
"""ram ev / opt ev""",1.328466
"""urs ev / opt ev""",0.449342
"""ram ev / opt mean""",0.949485
"""urs ev / opt mean""",0.321155


In [21]:
(df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg")).group_by("ip_right", "us_id_right","msg")
    .agg(pl.col(MEAN_NUM_XACT).mean()).sort(by =MEAN_NUM_XACT,descending=True)).collect()

ip_right,us_id_right,msg,mean_num_xact
u64,u64,str,f64
27638,12,"""32000""",1971.692625
38598,15,"""32000""",1971.692625
38598,14,"""32000""",1971.692625
28025,6,"""32000""",1971.692625
38598,8,"""32000""",1971.692625
…,…,…,…
33649,10,"""33322""",401.636313
33649,6,"""33331""",376.238875
33649,1,"""33323""",322.353875


In [5]:
#optimal strategy for each model of diffusion
opt = lf_opt.sort(pl.col("max_mean_num_xact"),descending = True).collect()
opt.write_parquet( "../../experiments/twitter_81306/opt.parquet")
opt

us_id,ip,argmax_msg,max_mean_num_xact
u64,u64,str,f64
14,38598,"""32000""",21271.88
4,38598,"""32000""",17740.36
4,28025,"""33000""",15809.76
15,28025,"""31000""",15723.89
13,38598,"""32000""",14421.54
…,…,…,…
13,33649,"""32331""",0.57
11,33649,"""32333""",0.48
2,33649,"""31311""",0.3


In [11]:
opt = lf_opt.sort(pl.col("max_mean_num_xact"),descending = True)
opt.filter(pl.col("argmax_msg").is_in(["30000","31000","32000","33000"])).group_by("argmax_msg").agg(pl.col("max_mean_num_xact").mean()).collect()

argmax_msg,max_mean_num_xact
str,f64
"""33000""",11304.415714
"""31000""",6598.578571
"""30000""",1244.31
"""32000""",9293.177333


In [8]:
###ram
lf_rob_details = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .sort(by="mean_num_xact", descending=True)
    # 上位5行を取得
).collect()
lf_rob_details

msg,mean_num_xact
str,f64
"""32000""",1971.692625
"""31000""",1957.660188
"""30000""",1950.243375
"""33000""",1906.779125
"""30100""",1754.464188
…,…
"""00233""",163.912813
"""02333""",144.857687
"""03333""",142.906688


In [6]:
lf_rob_details.select(pl.col("mean_num_xact").mean())

mean_num_xact
f64
666.908027


In [1]:
# read parquet file
import polars as pl
# user_analysis= pl.read_parquet("../../experiments/user_analysis.parquet")
user_analysis= pl.read_parquet("../../experiments/twitter_81306/user_analysis.parquet")
user_analysis


: 

In [32]:
user_analysis

ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users
u64,u64,str,u64,f64,f64
315,0,"""120""",0,0.85,0.86
315,0,"""120""",1,0.84,1.44
315,0,"""120""",2,0.99,0.75
315,0,"""120""",3,1.0,1.0
315,0,"""120""",4,1.0,2.52
…,…,…,…,…,…
163,1,"""102""",470,0.0,0.0
163,1,"""102""",471,0.0,0.72
163,1,"""102""",472,0.0,0.0


一人ずつ抽出したユーザーの外部行動回数

ロバストな戦略であるメッセージシーケンス :

"44000"	170.516875

"43000"	166.319688

"42000"	162.03375

"44100"	161.453937

"44001"	160.882688

In [9]:
ram_msg = (lf_rob_details.lazy().select(pl.col("msg")).collect())
ram_msg_list = ram_msg["msg"][:10].to_list()
# ram_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).collect()

reverse_msg_list = ram_msg["msg"][-10:].to_list()
# reverse_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(reverse_msg_list)).collect()


In [10]:
ram_msg_list

['32000',
 '31000',
 '30000',
 '33000',
 '30100',
 '31100',
 '22000',
 '23000',
 '32100',
 '33100']

In [11]:
reverse_msg_list

['13333',
 '02233',
 '01332',
 '01233',
 '00332',
 '00233',
 '02333',
 '03333',
 '01333',
 '00333']

In [10]:
ram_msg_us_an.lazy().filter(pl.col("num_share_of_users") > 4).collect()

ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users
u64,u64,str,u64,f64,f64
33655,0,"""31000""",2149,0.97,4.61
33655,0,"""31000""",17100,0.98,4.72
33655,0,"""31000""",18787,0.99,4.1
33655,0,"""31000""",36047,1.0,4.79
33655,0,"""31000""",40584,1.0,4.72
…,…,…,…,…,…
6257,9,"""32000""",55326,0.77,4.14
6257,9,"""32000""",79937,0.74,4.04
6257,9,"""32000""",80004,0.99,4.13


In [18]:
reverse_msg_us_an.write_parquet("../../experiments/reverse_msg_us_an.parquet")

In [19]:
ram_msg_us_an.write_parquet("../../experiments/ram_msg_us_an.parquet")

In [ ]:
ram_ratio = (user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list))
        .select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")))
ram_mean_xact_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"))
ram_mean_share_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"))
ram = (
    pl.concat([ram_mean_xact_of_users, ram_mean_share_of_users, ram_ratio], how="horizontal").collect())
ram


mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.262091,0.30411,0.861831


####今回のシミュレーションには0超過のユーザーしか抽出されていないため、正確な平均が反映されたコード

In [16]:
ram_msg = pl.read_parquet("../../experiments/twitter_81306/ram_msg_us_an.parquet")
ram_msg

ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users
u64,u64,str,u64,f64,f64
9199,0,"""31000""",1,0.0,0.03
9199,0,"""31000""",4,0.0,0.06
9199,0,"""31000""",9,0.02,0.0
9199,0,"""31000""",23,0.08,0.0
9199,0,"""31000""",26,0.02,0.0
…,…,…,…,…,…
68985,15,"""33100""",78535,0.08,0.01
68985,15,"""33100""",80119,0.12,0.0
68985,15,"""33100""",80265,0.34,0.17


In [17]:
action_stats_per_msg = (ram_msg.lazy().filter(pl.col("msg").is_in(ram_msg_list)).group_by("msg").agg(
    pl.col("num_xact_of_users").mean(),
    pl.col("num_share_of_users").mean())
)
action_stats_per_msg.collect()

msg,num_xact_of_users,num_share_of_users
str,f64,f64
"""32000""",0.27254,0.282888
"""30100""",0.255383,0.355378
"""32100""",0.272152,0.261161
"""30000""",0.249215,0.376328
"""23000""",0.251097,0.298533
"""31100""",0.266003,0.307411
"""31000""",0.261073,0.326456
"""33100""",0.277763,0.221196
"""33000""",0.277977,0.241252


In [20]:
ram =lf_rob_details[:10].join(action_stats_per_msg.collect(),left_on="msg", right_on="msg").sort(pl.col("mean_num_xact"), descending=True)

In [33]:
ram_ratio = ram.lazy().select(pl.col("msg"),pl.col("mean_num_xact"),pl.col("num_xact_of_users"),pl.col("num_share_of_users"),(pl.col("num_xact_of_users") / pl.col("num_share_of_users")).alias("xact_share_ratio")).collect()
ram_ratio 

msg,mean_num_xact,num_xact_of_users,num_share_of_users,xact_share_ratio
str,f64,f64,f64,f64
"""32000""",1971.692625,0.27254,0.282888,0.96342
"""31000""",1957.660188,0.261073,0.326456,0.799719
"""30000""",1950.243375,0.249215,0.376328,0.662227
"""33000""",1906.779125,0.277977,0.241252,1.152227
"""30100""",1754.464188,0.255383,0.355378,0.718623
"""31100""",1752.777625,0.266003,0.307411,0.865301
"""22000""",1742.086375,0.242693,0.344658,0.704154
"""23000""",1719.51125,0.251097,0.298533,0.841104
"""32100""",1712.298687,0.272152,0.261161,1.042086


In [45]:
ram_ratio_mean = ram_ratio.select(pl.col("mean_num_xact").mean(),pl.col("num_xact_of_users").mean(),pl.col("num_share_of_users").mean(),pl.col("xact_share_ratio").mean())
ram_ratio_mean

mean_num_xact,num_xact_of_users,num_share_of_users,xact_share_ratio
f64,f64,f64,f64
1815.804544,0.262589,0.301526,0.900459


In [ ]:
re_ram_msg = pl.read_parquet("../../experiments/twitter_81306/reverse_msg_us_an.parquet")
action_stats_per_msg = (re_ram_msg.lazy().filter(pl.col("msg").is_in(reverse_msg_list)).group_by("msg").agg(
    pl.col("num_xact_of_users").mean(),
    pl.col("num_share_of_users").mean())
)
ram =lf_rob_details.join(action_stats_per_msg.collect(),left_on="msg", right_on="msg").sort(pl.col("mean_num_xact"), descending=True)
re_ram_ratio = ram.lazy().select(pl.col("msg"),pl.col("mean_num_xact"),pl.col("num_xact_of_users"),pl.col("num_share_of_users"),(pl.col("num_xact_of_users") / pl.col("num_share_of_users")).alias("xact_share_ratio")).collect()
re_ram_ratio

msg,mean_num_xact,num_xact_of_users,num_share_of_users,xact_share_ratio
str,f64,f64,f64,f64
"""13333""",176.582812,0.200084,0.208078,0.96158
"""02233""",175.57825,0.114405,0.280044,0.408525
"""01332""",171.856437,0.101695,0.294872,0.344879
"""01233""",169.0435,0.08959,0.292721,0.306059
"""00332""",168.221312,0.078627,0.307673,0.255555
"""00233""",163.912813,0.069287,0.303262,0.228472
"""02333""",144.857687,0.12186,0.269813,0.451648
"""03333""",142.906688,0.151186,0.257636,0.586821
"""01333""",139.436688,0.094801,0.281407,0.336882


['30000',
 '31000',
 '32000',
 '30001',
 '20000',
 '22000',
 '21000',
 '33000',
 '31001',
 '30100']

In [46]:
re_ram_ratio_mean = re_ram_ratio.select(pl.col("mean_num_xact").mean(),pl.col("num_xact_of_users").mean(),pl.col("num_share_of_users").mean(),pl.col("xact_share_ratio").mean())
re_ram_ratio_mean

mean_num_xact,num_xact_of_users,num_share_of_users,xact_share_ratio
f64,f64,f64,f64
159.080519,0.109499,0.279096,0.412902


逆の順番に情報を提供すると

In [16]:
re_ram_xact_mean= user_analysis.lazy().filter(pl.col("msg") == "00033").select(pl.col("num_xact_of_users").mean())
re_ram_share_mean= user_analysis.lazy().filter(pl.col("msg") == "00033").select(pl.col("num_share_of_users").mean())

re_ram_xact_mean.collect()
re_ram_share_mean.collect()

num_share_of_users
f64
0.315331


最適戦略

拡散モデルip,us_idの組み合わせに、それに向ける最適解のmsgを同時に適用した全てのリストのユーザー状態

In [5]:
opt_df = (
    df.lazy()
    .filter(pl.col("mean_num_xact") == pl.col("mean_num_xact").max().over(["us_id", "ip"]))
    .unique(["ip", "us_id", "mean_num_xact"])
    .sort(pl.col("mean_num_xact"),descending = True)
    .select(pl.col("us_id"),pl.col("ip"),pl.col("msg"))
)
# 2. 結合の「キー」として使う LazyFrame を準備
#    opt_df の "argmax_msg" を user_analysis の "msg" に名前を合わせる
opt_keys = opt_df.lazy().select(
    pl.col("us_id"),
    pl.col("ip"),
    pl.col("msg")
)
# 3. user_analysis を 'semi' join でフィルタリングする
opt_ratio_lazy = user_analysis.lazy().join(
    opt_keys,
    on=["us_id", "ip", "msg"],  # 3つのキーがすべて一致する行を探す
    how="semi"                  # user_analysis 側に存在する行だけを残す
).collect()
opt_ratio_lazy


: 

In [7]:
opt_ratio_lazy = pl.read_parquet("../../experiments/twitter_81306/opt/opt_ratio_lazy.parquet")
opt_ratio_lazy

ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users
u64,u64,str,u64,f64,f64
9199,0,"""30011""",1,0.0,0.02
9199,0,"""30011""",4,0.0,0.14
9199,0,"""30011""",6,0.0,0.03
9199,0,"""30011""",7,0.03,0.07
9199,0,"""30011""",9,0.04,0.0
…,…,…,…,…,…
68985,15,"""30000""",79824,0.0,0.01
68985,15,"""30000""",80119,0.07,0.07
68985,15,"""30000""",80265,0.39,0.5


各拡散モデルにおけるOptimalのmsgの結果のユーザー状態

ユーザー1人の平均外部・内部行動回数と外部対内部の比率

In [5]:
opt_ratio = (opt_ratio_lazy.lazy().group_by(["msg","us_id","ip"])
        .agg(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"),
             pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"),
            (pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")
        )
        .collect())
opt_ratio

msg,us_id,ip,mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
str,u64,u64,f64,f64,f64
"""30010""",13,79973,0.134906,0.148113,0.910828
"""30000""",5,69023,0.06342,0.051303,1.23619
"""30000""",4,79973,0.047638,0.054216,0.878664
"""30002""",13,12605,0.02523,0.023032,1.095413
"""32221""",3,33649,0.45,0.71,0.633803
…,…,…,…,…,…
"""21113""",8,68985,0.226,0.247333,0.913747
"""33000""",11,38598,0.388505,0.373066,1.041383
"""30103""",9,9199,0.057205,0.092773,0.61661


最適解OPTとして扱われたmsgの頻度

In [2]:
opt_ratio_lazy=pl.read_parquet("../../experiments/twitter_81306/opt/opt_ratio_lazy.parquet")
opt_ratio = pl.read_parquet("../../experiments/twitter_81306/opt/opt_ratio.parquet")

In [9]:
opt_msg_counts =(
opt_ratio.lazy().group_by("msg").agg(pl.len().alias("msg_count")).sort("msg_count",descending=True).collect()
)
opt_msg_list = opt_msg_counts.select(pl.col("msg"))
opt_msg_counts
opt_msg_counts.write_csv("../test/opt_msg_list.csv")

In [138]:
ttt =opt_ratio.lazy().filter(pl.col("msg") == "44000")
# ,(pl.col("mean_num_share_of_users").mean()),(pl.col("mean_xact_of_users").mean() / pl.col("mean_share_of_users").mean()).alias("xact_share_ratio"))
# .collect()

ttt.mean().collect()

msg,us_id,ip,mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
str,f64,f64,f64,f64,f64
null,4.017857,224.678571,0.455427,0.492116,0.9829


In [152]:
opt = (opt_ratio_lazy.lazy().filter(pl.col("msg").is_in(opt_msg_list["msg"].to_list()))).collect()
opt.write_parquet("../../experiments/opt.parquet")

ロバストな戦略と最適戦略におけるユーザーの外部・内部行動回数の傾向

ロバストな戦略に該当する200の場合、ユーザー1人の外部行動回数の平均: 0.734416, ユーザー1人の内部行動回数の平均: 1.194905
        



最適戦略に該当する210の場合

ユーザー1人の外部行動回数の平均: 0.671784, 内部行動回数の平均:0.994447



両方の戦略とも内部行動の回数が常に外部行動の回数より上回っている

In [96]:
##opt
lf_opt.sort("max_mean_num_xact",descending=True).collect()

us_id,ip,argmax_msg,max_mean_num_xact
u64,u64,str,f64
8,130,"""44100""",376.36
8,246,"""44200""",365.85
8,113,"""44100""",364.65
1,113,"""44100""",346.54
7,113,"""44000""",346.45
…,…,…,…
5,329,"""42001""",21.33
0,62,"""41001""",17.61
3,62,"""41004""",14.88
